# Predictive Maintenance — Interim Project Notebook

**Learner:** Akram  
**Project:** Predictive Maintenance  
**Dataset:** `engine_data.csv`  
**Target:** `Engine Condition` (binary classification)

## Notebook purpose
This notebook implements the interim-project workflow: data registration and validation, exploratory data analysis (EDA), data preparation, model building, hyperparameter tuning, experiment tracking, evaluation, and saving the selected model.

> **Important business caveat:** The dataset labels are `0` and `1`, but the supplied dataset documentation does not define which label means failure or normal operation. Until the label meaning is confirmed with the dataset owner, this notebook refers to them as **Class 0** and **Class 1** and avoids interpreting either class as a confirmed failure event.

## Reproducibility
The notebook uses a fixed random seed, loads the CSV from the project `data/` folder when available (and falls back to the uploaded CSV path for this environment), writes processed datasets and model artifacts under `artifacts/`, and saves plots under `artifacts/figures/`.

## 1. Setup and project folders

**Methodology.** Import the required libraries, set a fixed random seed, create the expected project folders, and record the software versions. If the CSV is not already in `data/`, copy the provided source file there so the project has the required repository-style structure.

In [1]:
from pathlib import Path
import shutil
import warnings
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Prefer the checked-out project directory so the notebook works locally and in GitHub Codespaces.
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data" / "engine_data.csv").exists():
    PROJECT_DIR = CURRENT_DIR
elif CURRENT_DIR.name == "content" and str(CURRENT_DIR).startswith("/content"):
    PROJECT_DIR = CURRENT_DIR
else:
    PROJECT_DIR = Path("/tmp/Akram_PredictiveMaintenance")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_DIR / "data"

# Ensure the expected folder exists, then locate the user-provided CSV if it is not registered yet.
DATA_DIR.mkdir(parents=True, exist_ok=True)
REGISTERED_CSV = DATA_DIR / "engine_data.csv"

source_candidates = [
    PROJECT_DIR / "data" / "engine_data.csv",
    PROJECT_DIR / "engine_data.csv",
    CURRENT_DIR / "data" / "engine_data.csv",
    Path("/content/engine_data.csv"),
    Path("/mnt/data/engine_data.csv"),
    Path("/mnt/data/data/engine_data.csv"),
]
if not REGISTERED_CSV.exists():
    source_csv = next((p for p in source_candidates if p.exists() and p.resolve() != REGISTERED_CSV.resolve()), None)
    if source_csv is not None:
        shutil.copy2(source_csv, REGISTERED_CSV)

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FIGURE_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project directory:", PROJECT_DIR.resolve())
print("Registered dataset:", REGISTERED_CSV.resolve())
print("Dataset file exists:", REGISTERED_CSV.exists())
print("Random seed:", RANDOM_STATE)

Python: 3.13.5
Platform: Linux-6.18.44-x86_64-with-glibc2.41
Project directory: /tmp/Akram_PredictiveMaintenance
Registered dataset: /tmp/Akram_PredictiveMaintenance/data/engine_data.csv
Dataset file exists: True
Random seed: 42


## 2. Data registration and validation

**Methodology.** Load the registered CSV, check the expected columns, verify the target exists, summarize dimensions and data types, and check missing values, duplicate rows, and target labels. The validation fails explicitly if required columns are missing.

In [2]:
# Dataset availability check.
# If the file is missing in Google Colab, upload engine_data.csv when prompted.
from pathlib import Path
import shutil

REGISTERED_CSV = DATA_DIR / "engine_data.csv"
REGISTERED_CSV.parent.mkdir(parents=True, exist_ok=True)

if not REGISTERED_CSV.exists():
    repo_csv = Path.cwd() / "data" / "engine_data.csv"
    if repo_csv.exists():
        shutil.copy2(repo_csv, REGISTERED_CSV)

if not REGISTERED_CSV.exists():
    colab_upload_available = False
    try:
        from google.colab import files
        colab_upload_available = True
    except ImportError:
        pass

    if colab_upload_available:
        print("Please select the provided engine_data.csv file.")
        uploaded = files.upload()
        matching = next((name for name in uploaded if Path(name).name.lower() == "engine_data.csv"), None)
        if matching:
            shutil.move(matching, REGISTERED_CSV)
        else:
            raise FileNotFoundError("Upload engine_data.csv using the file picker.")
    else:
        raise FileNotFoundError(
            f"Dataset not found at {REGISTERED_CSV.resolve()}. "
            "Place engine_data.csv in the project's data/ folder and rerun."
        )

df = pd.read_csv(REGISTERED_CSV)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (19535, 7)


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp,Engine Condition
0,700,2.493592,11.790927,3.178981,84.144163,81.632187,1
1,876,2.941606,16.193866,2.464504,77.640934,82.445724,0
2,520,2.961746,6.553147,1.064347,77.752266,79.645777,1
3,473,3.707835,19.510172,3.727455,74.129907,71.774629,1
4,619,5.672919,15.738871,2.052251,78.396989,87.000225,0


In [3]:
EXPECTED_COLUMNS = [
    "Engine rpm",
    "Lub oil pressure",
    "Fuel pressure",
    "Coolant pressure",
    "lub oil temp",
    "Coolant temp",
    "Engine Condition",
]
TARGET = "Engine Condition"

df = pd.read_csv(REGISTERED_CSV)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

validation = {
    "file_exists": REGISTERED_CSV.exists(),
    "row_count": int(df.shape[0]),
    "column_count": int(df.shape[1]),
    "missing_expected_columns": missing_columns,
    "unexpected_columns": unexpected_columns,
    "missing_cells": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "target_present": TARGET in df.columns,
}
print("DATASET VALIDATION")
print(json.dumps(validation, indent=2))
assert not missing_columns, f"Dataset is missing required columns: {missing_columns}"
assert TARGET in df.columns, f"Target column '{TARGET}' not found."

print("\nDATA TYPES")
display(df.dtypes.rename("dtype").to_frame())
print("\nFIRST FIVE ROWS")
display(df.head())
print("\nNUMERIC SUMMARY")
display(df.describe().T.round(2))

DATASET VALIDATION
{
  "file_exists": true,
  "row_count": 19535,
  "column_count": 7,
  "missing_expected_columns": [],
  "unexpected_columns": [],
  "missing_cells": 0,
  "duplicate_rows": 0,
  "target_present": true
}

DATA TYPES


,dtype
Engine rpm,int64
Lub oil pressure,float64
Fuel pressure,float64
Coolant pressure,float64
lub oil temp,float64
Coolant temp,float64
Engine Condition,int64



FIRST FIVE ROWS


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp,Engine Condition
0,700,2.493592,11.790927,3.178981,84.144163,81.632187,1
1,876,2.941606,16.193866,2.464504,77.640934,82.445724,0
2,520,2.961746,6.553147,1.064347,77.752266,79.645777,1
3,473,3.707835,19.510172,3.727455,74.129907,71.774629,1
4,619,5.672919,15.738871,2.052251,78.396989,87.000225,0



NUMERIC SUMMARY


,count,mean,std,min,25%,50%,75%,max
Engine rpm,19535.0,791.24,267.61,61.00,593.00,746.00,934.00,2239.00
Lub oil pressure,19535.0,3.30,1.02,0.00,2.52,3.16,4.06,7.27
Fuel pressure,19535.0,6.66,2.76,0.00,4.92,6.20,7.74,21.14
Coolant pressure,19535.0,2.34,1.04,0.00,1.60,2.17,2.85,7.48
lub oil temp,19535.0,77.64,3.11,71.32,75.73,76.82,78.07,89.58
Coolant temp,19535.0,78.43,6.21,61.67,73.90,78.35,82.92,195.53
Engine Condition,19535.0,0.63,0.48,0.00,0.00,1.00,1.00,1.00


## 3. Exploratory Data Analysis (EDA)

### 3.1 Data overview and target distribution

**Methodology.** Review the dataset dimensions, summary statistics, target frequency, and target proportions. Class imbalance is assessed before selecting evaluation metrics and the train/test strategy.

In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Feature count: {df.shape[1] - 1}")
print(f"Missing cells: {int(df.isna().sum().sum())}")
print(f"Duplicate rows: {int(df.duplicated().sum())}")

target_counts = df[TARGET].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(df) * 100).round(2)
})
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Figure 1. Target Class Distribution")
ax.set_xlabel("Engine Condition class")
ax.set_ylabel("Record count")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_01_target_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

Rows: 19,535
Columns: 7
Feature count: 6
Missing cells: 0
Duplicate rows: 0


,count,percentage
Engine Condition,,
0,7218,36.95
1,12317,63.05


**Initial interpretation.** Class proportions help determine whether accuracy alone could hide poor performance on the less frequent class. The labels are kept as Class 0 and Class 1 because the source file does not provide a label dictionary.

### 3.2 Univariate analysis

**Methodology.** Plot a histogram for each sensor variable and inspect descriptive statistics. Box plots are used as a screening tool for extreme values; an extreme value is not automatically removed because it may represent a real operating condition.

In [5]:
FEATURES = [c for c in df.columns if c != TARGET]
display(df[FEATURES].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).T.round(2))

n_features = len(FEATURES)
ncols = 2
nrows = int(np.ceil(n_features / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.6 * nrows))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, FEATURES):
    ax.hist(df[col].dropna(), bins=35, edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
for ax in axes[len(FEATURES):]:
    ax.axis("off")
fig.suptitle("Figure 2. Sensor Feature Distributions", y=1.01, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_02_feature_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
df[FEATURES].plot(kind="box", ax=ax, showfliers=True)
ax.set_title("Figure 3. Sensor Feature Box Plots (Outlier Screening)")
ax.set_ylabel("Observed value (feature-specific units)")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_03_boxplots.png", dpi=160, bbox_inches="tight")
plt.show()

,count,mean,std,min,1%,25%,50%,75%,99%,max
Engine rpm,19535.0,791.24,267.61,61.00,380.00,593.00,746.00,934.00,1565.00,2239.00
Lub oil pressure,19535.0,3.30,1.02,0.00,0.86,2.52,3.16,4.06,5.60,7.27
Fuel pressure,19535.0,6.66,2.76,0.00,1.38,4.92,6.20,7.74,16.13,21.14
Coolant pressure,19535.0,2.34,1.04,0.00,0.73,1.60,2.17,2.85,5.95,7.48
lub oil temp,19535.0,77.64,3.11,71.32,73.41,75.73,76.82,78.07,87.37,89.58
Coolant temp,19535.0,78.43,6.21,61.67,65.74,73.90,78.35,82.92,91.78,195.53


**Interpretation guide.** Use the distribution plots to note skew, spread, and possible unusual readings. Because the columns use different physical units and scales, comparisons of raw magnitudes across different features are not meaningful. Any outlier treatment should be justified by domain knowledge rather than deleting observations solely because they appear extreme.

### 3.3 Bivariate analysis

**Methodology.** Compare each sensor's distribution across the two target classes using box plots and calculate class-wise means and medians. These comparisons are descriptive and do not by themselves establish causation.

In [6]:
class_summary = df.groupby(TARGET)[FEATURES].agg(["mean", "median", "std"]).round(2)
display(class_summary)

fig, axes = plt.subplots(3, 2, figsize=(13, 13))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    groups = [df.loc[df[TARGET] == label, col].dropna().values for label in sorted(df[TARGET].unique())]
    ax.boxplot(groups, labels=[str(x) for x in sorted(df[TARGET].unique())], showfliers=False)
    ax.set_title(col)
    ax.set_xlabel("Engine Condition class")
    ax.set_ylabel(col)
fig.suptitle("Figure 4. Sensor Measurements by Target Class", y=1.01, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_04_features_by_class.png", dpi=160, bbox_inches="tight")
plt.show()

mean_by_class = df.groupby(TARGET)[FEATURES].mean().T
mean_by_class.columns = [f"Class {c} mean" for c in mean_by_class.columns]
mean_by_class["absolute_mean_difference"] = (mean_by_class.iloc[:, 0] - mean_by_class.iloc[:, 1]).abs()
display(mean_by_class.sort_values("absolute_mean_difference", ascending=False).round(3))

Engine rpm                 ... Coolant temp             
                       mean median     std  ...         mean median   std
Engine Condition                            ...                          
0                     885.0  843.0  271.72  ...        78.80  78.77  5.97
1                     736.3  690.0  249.30  ...        78.21  78.10  6.33

[2 rows x 18 columns]

,Class 0 mean,Class 1 mean,absolute_mean_difference
Engine rpm,884.995,736.297,148.698
Fuel pressure,6.236,6.901,0.665
lub oil temp,78.024,77.420,0.604
Coolant temp,78.803,78.207,0.596
Lub oil pressure,3.222,3.351,0.129
Coolant pressure,2.368,2.316,0.052


**Business interpretation.** Features whose distributions differ between classes may provide predictive signal. The size and consistency of differences should be considered alongside model performance and data quality; class-wise differences alone are not evidence that a sensor causes a particular engine condition.

### 3.4 Multivariate analysis

**Methodology.** Use a correlation heatmap to inspect linear relationships among sensor variables and the target, then compare selected pairs of sensor variables with points colored by target class. Correlation is descriptive and may not capture nonlinear relationships.

In [7]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Figure 5. Correlation Matrix")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_05_correlation_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

# Target-colored pairwise scatter plots (sampled for readability).
pair_features = ["Engine rpm", "Lub oil pressure", "Fuel pressure"]
pair_df = df[pair_features + [TARGET]].sample(n=min(1500, len(df)), random_state=RANDOM_STATE)
classes = sorted(pair_df[TARGET].dropna().unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (x_col, y_col) in zip(axes, [
    ("Engine rpm", "Lub oil pressure"),
    ("Engine rpm", "Fuel pressure"),
    ("Lub oil pressure", "Fuel pressure"),
]):
    for cls in classes:
        subset = pair_df[pair_df[TARGET] == cls]
        ax.scatter(subset[x_col], subset[y_col], s=12, alpha=0.35, label=f"Class {cls}")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f"{x_col} vs {y_col}")
axes[0].legend(title="Target class")
fig.suptitle("Figure 6. Pairwise Sensor Relationships by Target Class")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_06_pairwise_sensors_by_class.png", dpi=160, bbox_inches="tight")
plt.show()

corr_with_target = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
display(corr_with_target.rename("correlation_with_target").to_frame().round(3))

,correlation_with_target
Engine rpm,-0.268
Fuel pressure,0.116
lub oil temp,-0.094
Lub oil pressure,0.061
Coolant temp,-0.046
Coolant pressure,-0.024


### 3.5 EDA findings and business implications

**Methodology.** Summarize quantitative checks and rank sensor features by their absolute correlation with the target. This ranking is an exploratory signal only; model-based validation is required before deciding which signals are operationally useful.

In [8]:
eda_findings = pd.DataFrame([
    ["Dataset size", f"{len(df):,} rows × {df.shape[1]} columns", "Provides the initial data volume for analysis."],
    ["Missing values", f"{int(df.isna().sum().sum()):,}", "No missing-value imputation is needed for the current source data."],
    ["Duplicate rows", f"{int(df.duplicated().sum()):,}", "No exact duplicate rows were detected."],
    ["Target classes", ", ".join(f"{k}: {v:,} ({v/len(df)*100:.2f}%)" for k, v in target_counts.items()), "Use stratified splitting and class-sensitive metrics."],
    ["Label definition", "Not supplied with CSV", "Confirm what Class 0 and Class 1 mean before operational interpretation."],
], columns=["Check", "Finding", "Potential implication"])
display(eda_findings)

print("EDA priority notes:")
print("- Confirm target label semantics with the dataset owner.")
print("- Review high/low sensor values with a domain expert before treating them as errors.")
print("- Use recall, precision, F1-score, ROC-AUC, and a confusion matrix in model evaluation.")
print("- Treat correlations as descriptive, not causal.")

,Check,Finding,Potential implication
0,Dataset size,"19,535 rows × 7 columns",Provides the initial data volume for analysis.
1,Missing values,0,No missing-value imputation is needed for the ...
2,Duplicate rows,0,No exact duplicate rows were detected.
3,Target classes,"0: 7,218 (36.95%), 1: 12,317 (63.05%)",Use stratified splitting and class-sensitive m...
4,Label definition,Not supplied with CSV,Confirm what Class 0 and Class 1 mean before o...


EDA priority notes:
- Confirm target label semantics with the dataset owner.
- Review high/low sensor values with a domain expert before treating them as errors.
- Use recall, precision, F1-score, ROC-AUC, and a confusion matrix in model evaluation.
- Treat correlations as descriptive, not causal.


## 4. Data preparation

**Methodology.** Separate the predictors and target, remove exact duplicate rows if any are found, validate the target, and split into training and test partitions using stratification. No columns are removed because the supplied schema contains only six candidate sensor features and one target. The split files are saved as workflow artifacts for downstream jobs.

In [9]:
# Start from the registered source and apply transparent, minimal cleaning.
clean_df = df.copy()
before_rows = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
removed_duplicates = before_rows - len(clean_df)

# Validate the target and expected features.
assert clean_df[TARGET].notna().all(), "Target contains missing values."
assert clean_df[TARGET].nunique() >= 2, "Target must contain at least two classes."
assert all(col in clean_df.columns for col in FEATURES), "One or more expected features are missing."

X = clean_df[FEATURES].copy()
y = clean_df[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Persist artifacts so a later workflow job can consume them.
train_df = X_train.copy()
train_df[TARGET] = y_train
test_df = X_test.copy()
test_df[TARGET] = y_test

train_path = ARTIFACT_DIR / "train.csv"
test_path = ARTIFACT_DIR / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

prep_summary = pd.DataFrame({
    "Dataset": ["Cleaned full dataset", "Training split", "Test split"],
    "Rows": [len(clean_df), len(X_train), len(X_test)],
    "Features": [len(FEATURES), len(FEATURES), len(FEATURES)],
    "Target included": ["Yes", "Yes", "Yes"],
})
display(prep_summary)
print(f"Exact duplicate rows removed: {removed_duplicates}")
print(f"Training class proportions:\n{y_train.value_counts(normalize=True).sort_index().round(4)}")
print(f"Test class proportions:\n{y_test.value_counts(normalize=True).sort_index().round(4)}")
print("Saved:", train_path)
print("Saved:", test_path)

,Dataset,Rows,Features,Target included
0,Cleaned full dataset,19535,6,Yes
1,Training split,15628,6,Yes
2,Test split,3907,6,Yes


Exact duplicate rows removed: 0
Training class proportions:
Engine Condition
0    0.3695
1    0.6305
Name: proportion, dtype: float64
Test class proportions:
Engine Condition
0    0.3696
1    0.6304
Name: proportion, dtype: float64
Saved: /tmp/Akram_PredictiveMaintenance/artifacts/train.csv
Saved: /tmp/Akram_PredictiveMaintenance/artifacts/test.csv


### 4.1 Re-load the workflow artifacts

This cell deliberately reloads the saved CSV files, demonstrating the handoff between data preparation and model-building stages. In a CI workflow, these files can be published as workflow artifacts and downloaded by the model-training job.

In [10]:
train_loaded = pd.read_csv(train_path)
test_loaded = pd.read_csv(test_path)

X_train = train_loaded[FEATURES]
y_train = train_loaded[TARGET].astype(int)
X_test = test_loaded[FEATURES]
y_test = test_loaded[TARGET].astype(int)

print("Loaded workflow artifacts successfully.")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train target distribution:", y_train.value_counts().sort_index().to_dict())
print("Test target distribution:", y_test.value_counts().sort_index().to_dict())

Loaded workflow artifacts successfully.
X_train: (15628, 6) | X_test: (3907, 6)
Train target distribution: {0: 5774, 1: 9854}
Test target distribution: {0: 1444, 1: 2463}


## 5. Model building and experimentation tracking

**Methodology.** Establish a majority-class dummy baseline, then tune four candidate classifiers: Decision Tree, Random Forest, AdaBoost, and Gradient Boosting. Grid search uses stratified three-fold cross-validation and F1-score as the primary selection metric. All tested parameter combinations, mean cross-validation scores, and best parameters are saved for traceability.

**Model-selection rationale.** F1-score balances precision and recall. Since the business cost of false negatives versus false positives has not been specified, the final model should be reviewed against the confusion matrix and operational cost assumptions before deployment.

In [11]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# A simple baseline establishes a reference point.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_metrics = {
    "model": "DummyClassifier (majority class)",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision_weighted": precision_score(y_test, baseline_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, baseline_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, baseline_pred, average="weighted", zero_division=0),
}
print("Baseline results:")
display(pd.DataFrame([baseline_metrics]).round(4))

model_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        {
            "max_depth": [8, None],
            "min_samples_leaf": [1],
            "criterion": ["gini"],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced"),
        {
            "n_estimators": [20, 40],
            "max_depth": [None],
            "min_samples_leaf": [1],
        },
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [20, 40],
            "learning_rate": [1.0],
        },
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [20, 40],
            "learning_rate": [0.1],
            "max_depth": [2],
        },
    ),
}

experiment_rows = []
best_estimators = {}
search_objects = {}

for model_name, (estimator, param_grid) in model_grids.items():
    print(f"Training and tuning: {model_name}")
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="f1_weighted",
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=True,
        verbose=0,
    )
    start = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - start
    search_objects[model_name] = search
    best_estimators[model_name] = search.best_estimator_

    results = pd.DataFrame(search.cv_results_)
    for _, row in results.iterrows():
        params = row["params"]
        experiment_rows.append({
            "model": model_name,
            "params": json.dumps(params, sort_keys=True),
            "mean_cv_f1_weighted": row["mean_test_score"],
            "std_cv_f1_weighted": row["std_test_score"],
            "mean_train_f1_weighted": row["mean_train_score"],
            "rank": int(row["rank_test_score"]),
            "fit_time_seconds_mean": row["mean_fit_time"],
            "search_elapsed_seconds": elapsed,
            "is_best_for_model": bool(row["rank_test_score"] == 1),
        })
    print(f"  Best CV weighted F1: {search.best_score_:.4f}")
    print(f"  Best parameters: {search.best_params_}")
    print(f"  Search elapsed: {elapsed:.1f} seconds")

experiments = pd.DataFrame(experiment_rows).sort_values(
    ["mean_cv_f1_weighted", "std_cv_f1_weighted"], ascending=[False, True]
).reset_index(drop=True)
experiments_path = ARTIFACT_DIR / "experiment_tracking.csv"
experiments.to_csv(experiments_path, index=False)

print("\nTop 10 experiment configurations:")
display(experiments.head(10).round(4))
print("Full experiment log saved to:", experiments_path)

Baseline results:


,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,DummyClassifier (majority class),0.6304,0.3974,0.6304,0.4875


Training and tuning: Decision Tree


  Best CV weighted F1: 0.6078
  Best parameters: {'criterion': 'gini', 'max_depth': 8, 'min_samples_leaf': 1}
  Search elapsed: 0.6 seconds
Training and tuning: Random Forest


  Best CV weighted F1: 0.6350
  Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 40}
  Search elapsed: 5.9 seconds
Training and tuning: AdaBoost


  Best CV weighted F1: 0.6462
  Best parameters: {'learning_rate': 1.0, 'n_estimators': 20}
  Search elapsed: 2.1 seconds
Training and tuning: Gradient Boosting


  Best CV weighted F1: 0.6375
  Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 40}
  Search elapsed: 3.3 seconds

Top 10 experiment configurations:


,model,params,mean_cv_f1_weighted,std_cv_f1_weighted,mean_train_f1_weighted,rank,fit_time_seconds_mean,search_elapsed_seconds,is_best_for_model
0,AdaBoost,"{""learning_rate"": 1.0, ""n_estimators"": 20}",0.6462,0.0049,0.6521,1,0.1977,2.1429,True
1,AdaBoost,"{""learning_rate"": 1.0, ""n_estimators"": 40}",0.6451,0.0015,0.6523,2,0.3822,2.1429,False
2,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",0.6375,0.0053,0.6466,1,0.5250,3.3079,True
3,Random Forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.6350,0.0015,0.9997,1,0.8592,5.9148,True
4,Random Forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.6326,0.0053,0.9977,2,0.4621,5.9148,False
5,Decision Tree,"{""criterion"": ""gini"", ""max_depth"": 8, ""min_sam...",0.6078,0.0056,0.6741,1,0.0577,0.5616,True
6,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",0.6036,0.0044,0.6109,2,0.2799,3.3079,False
7,Decision Tree,"{""criterion"": ""gini"", ""max_depth"": null, ""min_...",0.5877,0.0028,1.0000,2,0.0868,0.5616,False


Full experiment log saved to: /tmp/Akram_PredictiveMaintenance/artifacts/experiment_tracking.csv


### 5.0.1 Best configuration by candidate model

The table below summarizes the best cross-validation configuration for each candidate model. The full configuration-level log remains available in `artifacts/experiment_tracking.csv`.

In [12]:
best_config_rows = []
for model_name, search in search_objects.items():
    best_config_rows.append({
        "model": model_name,
        "best_cv_weighted_f1": search.best_score_,
        "best_parameters": json.dumps(search.best_params_, sort_keys=True),
        "cv_folds": cv.get_n_splits(),
    })
best_config_summary = pd.DataFrame(best_config_rows).sort_values(
    "best_cv_weighted_f1", ascending=False
).reset_index(drop=True)
display(best_config_summary.round(4))

,model,best_cv_weighted_f1,best_parameters,cv_folds
0,AdaBoost,0.6462,"{""learning_rate"": 1.0, ""n_estimators"": 20}",3
1,Gradient Boosting,0.6375,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",3
2,Random Forest,0.6350,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",3
3,Decision Tree,0.6078,"{""criterion"": ""gini"", ""max_depth"": 8, ""min_sam...",3


### 5.1 Compare tuned models on the held-out test set

**Methodology.** Each candidate's best configuration is selected using stratified cross-validation on the training partition. The held-out test partition is then used to report final performance metrics only. The test scores must not be used to select the winning model. The dummy classifier is included as a reference baseline, not as a candidate for final model selection.

In [13]:
evaluation_rows = []
test_predictions = {}
test_probabilities = {}

for model_name, estimator in best_estimators.items():
    pred = estimator.predict(X_test)
    test_predictions[model_name] = pred

    row = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_weighted": precision_score(y_test, pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
    }
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X_test)[:, 1]
        test_probabilities[model_name] = proba
        row["roc_auc"] = roc_auc_score(y_test, proba)
    else:
        row["roc_auc"] = np.nan
    evaluation_rows.append(row)

# Add the dummy baseline to the same comparison table for context.
baseline_row = {
    "model": baseline_metrics["model"],
    "accuracy": baseline_metrics["accuracy"],
    "precision_weighted": baseline_metrics["precision_weighted"],
    "recall_weighted": baseline_metrics["recall_weighted"],
    "f1_weighted": baseline_metrics["f1_weighted"],
    "f1_macro": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
    "roc_auc": np.nan,
}
evaluation = pd.concat([pd.DataFrame(evaluation_rows), pd.DataFrame([baseline_row])], ignore_index=True)
evaluation = evaluation.sort_values("f1_weighted", ascending=False).reset_index(drop=True)
evaluation_path = ARTIFACT_DIR / "model_evaluation.csv"
evaluation.to_csv(evaluation_path, index=False)

display(evaluation.round(4))
print("Evaluation table saved to:", evaluation_path)

# Select the final candidate using cross-validation performance on the training data only.
# The held-out test set is used for final evaluation, not for choosing the model.
best_cv_row = best_config_summary.sort_values(
    ["best_cv_weighted_f1", "model"], ascending=[False, True]
).iloc[0]
best_model_name = best_cv_row["model"]
best_model = best_estimators[best_model_name]
best_pred = test_predictions[best_model_name]

print("Selected model using training-only cross-validation weighted F1:", best_model_name)
print(f"Mean CV weighted F1: {best_cv_row['best_cv_weighted_f1']:.4f}")
print("Held-out test metrics are reported below for final evaluation only.")

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro,roc_auc
0,AdaBoost,0.6542,0.6384,0.6542,0.6385,0.6002,0.6846
1,Gradient Boosting,0.6660,0.6499,0.6660,0.6336,0.5861,0.6981
2,Random Forest,0.6424,0.6254,0.6424,0.6266,0.5873,0.6645
3,Decision Tree,0.6063,0.6449,0.6063,0.6131,0.6009,0.6537
4,DummyClassifier (majority class),0.6304,0.3974,0.6304,0.4875,0.3867,NaN


Evaluation table saved to: /tmp/Akram_PredictiveMaintenance/artifacts/model_evaluation.csv
Selected model using training-only cross-validation weighted F1: AdaBoost
Mean CV weighted F1: 0.6462
Held-out test metrics are reported below for final evaluation only.


### 5.2 Final model evaluation plots

**Methodology.** The selected model is the candidate with the highest mean cross-validation weighted F1 on the training data. The held-out test set is used only to produce the final confusion matrix, ROC curve, and classification report. These plots help reviewers inspect error types and discrimination performance without using test results to choose a model.

In [14]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, ax=ax, values_format="d", colorbar=False
)
ax.set_title(f"Figure 7. Confusion Matrix — {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_07_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

if best_model_name in test_probabilities:
    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_test, test_probabilities[best_model_name], ax=ax)
    ax.set_title(f"Figure 8. ROC Curve — {best_model_name}")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "figure_08_roc_curve.png", dpi=160, bbox_inches="tight")
    plt.show()

print("Classification report for selected model:")
print(classification_report(y_test, best_pred, digits=4, zero_division=0))

Classification report for selected model:


              precision    recall  f1-score   support

           0     0.5453    0.3878    0.4533      1444
           1     0.6931    0.8104    0.7471      2463

    accuracy                         0.6542      3907
   macro avg     0.6192    0.5991    0.6002      3907
weighted avg     0.6384    0.6542    0.6385      3907



### 5.3 Feature importance

**Methodology.** If the selected estimator exposes `feature_importances_`, rank features by the model's internal importance score. These scores indicate how the fitted model used the features; they are not causal effects and may be affected by correlated predictors.

In [15]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    importance_df = importances.rename("importance").reset_index().rename(columns={"index": "feature"})
    display(importance_df.round(4))

    fig, ax = plt.subplots(figsize=(8, 5))
    importances.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Figure 9. Feature Importance — {best_model_name}")
    ax.set_xlabel("Importance score")
    ax.set_ylabel("Sensor feature")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "figure_09_feature_importance.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print(f"{best_model_name} does not expose feature_importances_.")

,feature,importance
0,Engine rpm,0.6194
1,Fuel pressure,0.1233
2,lub oil temp,0.0979
3,Lub oil pressure,0.0878
4,Coolant pressure,0.0417
5,Coolant temp,0.0300


### 5.4 Save the best model and model metadata

**Methodology.** Persist the best-performing tuned estimator, its feature list, target name, selection metric, and evaluation metrics. The saved artifacts can be consumed by a later deployment or repository-commit workflow.

In [16]:
model_path = ARTIFACT_DIR / "best_predictive_maintenance_model.joblib"
metadata_path = ARTIFACT_DIR / "best_model_metadata.json"

joblib.dump(best_model, model_path)

metadata = {
    "project": "Predictive Maintenance",
    "dataset_file": "data/engine_data.csv",
    "target": TARGET,
    "features": FEATURES,
    "selected_model": best_model_name,
    "selection_metric": "mean stratified 3-fold cross-validation weighted F1 on training data",
    "random_state": RANDOM_STATE,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "best_parameters": best_model.get_params(),
    "test_metrics": evaluation.loc[evaluation["model"] == best_model_name].iloc[0].to_dict(),
    "important_caveat": "Target label semantics were not provided; confirm whether Class 0/1 map to failure or normal operation before operational use."
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved best model:", model_path)
print("Saved metadata:", metadata_path)
print("Artifact directory contents:")
for path in sorted(ARTIFACT_DIR.rglob("*")):
    if path.is_file():
        print(f"- {path.relative_to(PROJECT_DIR)} ({path.stat().st_size:,} bytes)")

Saved best model: /tmp/Akram_PredictiveMaintenance/artifacts/best_predictive_maintenance_model.joblib
Saved metadata: /tmp/Akram_PredictiveMaintenance/artifacts/best_model_metadata.json
Artifact directory contents:
- artifacts/best_model_metadata.json (1,014 bytes)
- artifacts/best_predictive_maintenance_model.joblib (14,564 bytes)
- artifacts/experiment_tracking.csv (1,592 bytes)
- artifacts/figures/figure_01_target_distribution.png (34,432 bytes)
- artifacts/figures/figure_02_feature_distributions.png (151,977 bytes)
- artifacts/figures/figure_03_boxplots.png (84,309 bytes)
- artifacts/figures/figure_04_features_by_class.png (131,939 bytes)
- artifacts/figures/figure_05_correlation_matrix.png (135,090 bytes)
- artifacts/figures/figure_06_pairwise_sensors_by_class.png (448,651 bytes)
- artifacts/figures/figure_07_confusion_matrix.png (24,859 bytes)
- artifacts/figures/figure_08_roc_curve.png (56,260 bytes)
- artifacts/figures/figure_09_feature_importance.png (44,844 bytes)
- artifacts

## 6. Interim findings and business implications

This section is generated from the measured results above. Validate the target label definitions with the data owner before translating Class 0/Class 1 into failure or normal-condition language.

1. **Data quality:** the validation outputs show the dataset dimensions, missingness, duplicates, and schema status. Any unexpected values should be reviewed with an engine/domain specialist.
2. **Predictive signal:** EDA compares sensor distributions and correlations; tuned model performance provides a stronger test of whether the signals are useful together. The final model is selected using training-only cross-validation; the held-out test set is reserved for final evaluation.
3. **Operational trade-off:** review false negatives and false positives in the confusion matrix for the CV-selected model. If missing an impending failure is more costly than a false alarm, model selection should prioritize the corresponding class recall and be tied to a cost-based threshold.
4. **Deployment readiness:** the saved model and metadata are technical artifacts, not proof of production readiness. Before deployment, confirm label semantics, validate on representative future data, establish monitoring for drift, and agree on alert thresholds with maintenance stakeholders.
5. **Reproducibility:** the registered dataset, split files, experiment log, evaluation table, plots, and model artifact are saved under `data/` and `artifacts/`.

## 7. Appendix: artifacts, provenance, and workflow notes

### Dataset provenance
The notebook uses the user-provided `engine_data.csv`. No external datasets or images are downloaded.

### Saved artifacts
- `data/engine_data.csv` — registered source data
- `artifacts/train.csv`, `artifacts/test.csv` — stratified split artifacts
- `artifacts/experiment_tracking.csv` — all grid-search configurations and scores
- `artifacts/model_evaluation.csv` — held-out evaluation metrics
- `artifacts/best_predictive_maintenance_model.joblib` — selected fitted estimator
- `artifacts/best_model_metadata.json` — model metadata
- `artifacts/figures/` — numbered report-ready plots

### Workflow artifact handoff
For a GitHub Actions workflow, upload `artifacts/train.csv` and `artifacts/test.csv` from the data-preparation job as workflow artifacts, then download them in the model-training job. The final model and metadata can be committed or uploaded by a separate authorized pipeline step. This notebook saves the files locally but does not itself configure or execute GitHub Actions.


### 7.1 Example GitHub Actions artifact handoff

The notebook writes `artifacts/train.csv` and `artifacts/test.csv`. The following workflow example shows how one job can upload these split files and a later model-training job can download them. Save this as `.github/workflows/predictive-maintenance.yml` in the repository and adapt the job commands to the repository's actual scripts.

```yaml
name: Predictive Maintenance Interim Pipeline

on:
  workflow_dispatch:
  push:
    paths:
      - "data/**"
      - "src/**"
      - ".github/workflows/predictive-maintenance.yml"

jobs:
  prepare-data:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pandas numpy scikit-learn matplotlib joblib
      - run: python src/prepare_data.py
      - uses: actions/upload-artifact@v4
        with:
          name: prepared-data
          path: |
            artifacts/train.csv
            artifacts/test.csv

  train-model:
    needs: prepare-data
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pandas numpy scikit-learn matplotlib joblib
      - uses: actions/download-artifact@v4
        with:
          name: prepared-data
          path: artifacts
      - run: python src/train_model.py
      - uses: actions/upload-artifact@v4
        with:
          name: trained-model
          path: |
            artifacts/best_predictive_maintenance_model.joblib
            artifacts/best_model_metadata.json
```

**Scope note:** This notebook demonstrates the split-file handoff locally and documents the GitHub Actions pattern. The workflow above is an example and is not executed by running this notebook; the repository must contain the referenced scripts before the workflow can run successfully.